# 🎻 Mara's road: learning the best path by playing

*Policy gradient methods, Part II, Day 3. A coding exercise.*

Mara is a busker, a street fiddler. At dawn she stands in Eagle Pass with an empty hat. Today she walks down to the sea. At every place the road forks, southwest or southeast, and she plays in the next place she reaches. Whoever passes drops a few coins in her hat. A day is three roads long, so there are eight possible paths from Eagle Pass to the sea. Luck plays a part: every time Mara arrives somewhere, a few passers-by give one coin more or less than the usual pay of that place, so two days on the same path can pay a little differently.

You played this game in the browser. Now the computer plays it and learns, from the coins of its own days, which path to follow.

### The table

Mara chooses a road by looking at a table: one row for every place that has a choice (seven rows) and one column for every road (two columns). Every cell holds a *knob*, any real number, and every knob starts at 0. A softmax turns the two knobs of a row into two chances that add up to 1. With all knobs at 0 the chances are 0.5 and 0.5: Mara tosses a fair coin at every fork.

### The method

Repeat:

1. pick one cell of the table at random;
2. play N days (`n_days`) with the table, and N days with a copy of the table that has `eps` added to that cell. Both get the same dice. `J` is the average coins of the first N days and `J'` the average of the second;
3. measure the gradient of the cell: `g = (J' - J) / eps`;
4. move the cell: `cell = cell + alpha * g`.

This is the grasshopper's climb from the recap: the chosen cell is the grasshopper, and the average coins of a day is its altitude. Adding `eps` to the cell is the small step that measures the slope, and `alpha` is how far the grasshopper jumps. `eps` and `alpha` are two different small numbers.

### What you build

Four short pieces of code: **1.** `softmax`, **2.** `play_days`, **3.** `measured_gradient` and **4.** `learn`, the loop. Pieces 2, 3 and 4 come with most of their lines written, and in every piece you complete the lines marked with 🎯, eight in all. After each piece a check cell says in plain words whether it works. At the end you run the method and watch the average coins climb.

Run the cells from the top, one after another (click a cell and press Shift and Enter). Everywhere you see 🎯 it is your turn, and everything else runs by itself. A "Not yet." from a check is normal while a piece is unfinished, and so is a red box: read the message, fix the piece, and run the cells again.

Start with the cell below. It loads the tools and the checkers, and its code is hidden because nothing in it needs your attention.

In [ ]:
# @title Run this cell first (it loads numpy, the exact average coins and the checkers) { display-mode: "form" }

import inspect
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import LinearSegmentedColormap

BULLSEYE = "\U0001F3AF"

# ======================================================================
# The exact average coins of a table of knobs.
# It draws the plot and runs the checks. The method itself never calls it.
# The luck of every arrival averages to zero, so the usual coins are all it needs.
# ======================================================================

ALL_PATHS = [(a, b, c) for a in (0, 1) for b in (0, 1) for c in (0, 1)]   # the eight paths, as three roads


def places_of(path):
    # the three places a path visits. The roads out of place i lead to places 2i+1 and 2i+2.
    place = 0
    places = []
    for road in path:
        place = 2 * place + 1 + road
        places.append(place)
    return places


def first_road_chances(w):
    # the chance of road 0 in every row. For two knobs the softmax is 1 / (1 + exp(w2 - w1)).
    w = np.asarray(w, dtype=float)
    return 1.0 / (1.0 + np.exp(w[:, 1] - w[:, 0]))


def path_probabilities(w):
    # the chance of each of the eight paths: multiply the chances of its three roads
    first = first_road_chances(w)
    probabilities = []
    for path in ALL_PATHS:
        place = 0
        probability = 1.0
        for road in path:
            probability = probability * (first[place] if road == 0 else 1.0 - first[place])
            place = 2 * place + 1 + road
        probabilities.append(probability)
    return np.array(probabilities)


def path_coins():
    # the usual coins of each of the eight paths, which are also their average pay
    return np.array([sum(int(COINS[p]) for p in places_of(path)) for path in ALL_PATHS])


def exact_J(w):
    # the exact average coins of a day: every path weighted by its chance
    return float(path_probabilities(w) @ path_coins())


def most_likely_path(w):
    probabilities = path_probabilities(w)
    k = int(probabilities.argmax())
    return ALL_PATHS[k], float(probabilities[k])


# ======================================================================
# Printing the road
# ======================================================================

def show_road():
    print("place  name                 usual coins   roads out of it (road 0, road 1)")
    for i, name in enumerate(PLACES):
        roads = "%d, %d" % (2 * i + 1, 2 * i + 2) if i < 7 else "the day ends here"
        print("%5d  %-20s %11d   %s" % (i, name, COINS[i], roads))


def show_path(places):
    names = ["%s (%d)" % (PLACES[p], COINS[p]) for p in places]
    total = sum(int(COINS[p]) for p in places)
    print(" -> ".join(["Eagle Pass"] + names) + "   =   %d coins on average" % total)


# ======================================================================
# The checkers. Each one calls your function on small cases and tells you what it finds.
# ======================================================================

_passed = {}


def _say(n, ok, good, bad=""):
    _passed[n] = ok
    print(("Right. " + good) if ok else ("Not yet. " + bad))


def _blanks(function):
    # the lines marked with the bullseye that still hold three dots
    try:
        lines = inspect.getsource(function).splitlines()
    except (OSError, TypeError):
        return []
    tags = []
    for line in lines:
        code, hash_sign, comment = line.partition("#")
        if hash_sign and BULLSEYE in comment and "..." in code:
            tags.append(comment.replace(BULLSEYE, "").strip().split(")")[0])
    return tags


def _call(n, name, function, *args, hint=""):
    tags = _blanks(function)
    if tags:
        _say(n, False, "", "%s still has %s on the lines marked %s %s. Replace each ... with your code, run the cell of the piece again, and then run this check again."
             % (name, "a blank" if len(tags) == 1 else "blanks", BULLSEYE, ", ".join(tags)))
        return False, None
    try:
        return True, function(*args)
    except Exception as error:
        _say(n, False, "", "%s stopped with %s: %s. Look at the lines marked %s in the cell above. %s"
             % (name, type(error).__name__, error, BULLSEYE, hint))
        return False, None


def _plain(value):
    # numpy scalars print as np.float64(...) in recent versions, so show plain Python numbers
    try:
        return np.asarray(value).tolist()
    except Exception:
        return value


def _numbers(n, name, value, length):
    try:
        array = np.asarray(value, dtype=float)
    except (TypeError, ValueError):
        _say(n, False, "", "%s returned %r, which cannot be read as numbers. Is every line of the piece finished?" % (name, value))
        return None
    if array.shape != (length,):
        _say(n, False, "", "%s has to return %d numbers here, one per knob or day. It returned an array of shape %s." % (name, length, array.shape))
        return None
    return array


def _one_number(n, name, value):
    # one number, or None after saying why the value is not one
    try:
        array = np.asarray(value, dtype=float)
    except (TypeError, ValueError):
        _say(n, False, "", "%s returned %r, which cannot be read as a number. Is every line of the piece finished?" % (name, value))
        return None
    if array.shape != ():
        _say(n, False, "", "%s has to return one number, g. It returned an array of shape %s. Each J is an average over the days: use .mean()." % (name, array.shape))
        return None
    return float(array)


def _trusted_days(w, dice):
    # the answer key for play_days: three road dice and three luck draws a day
    first = first_road_chances(w)
    totals = []
    for row in dice:
        place, total = 0, 0
        for step in range(3):
            road = 0 if row[step] < first[place] else 1
            place = 2 * place + 1 + road
            total += int(COINS[place]) + luck_from(row[3 + step])
        totals.append(total)
    return np.array(totals)


def _names(places):
    names = [PLACES[p] for p in places]
    return ", ".join(names[:-1]) + " and " + names[-1]


def check_softmax(function):
    for knobs in [(0.0, 0.0), (2.0, 0.0), (-1.0, 3.0), (5.0, -5.0)]:
        ok, got = _call(1, "softmax", function, np.array(knobs), hint="np.exp works on a whole array, and .sum() adds its numbers up.")
        if not ok:
            return
        got = _numbers(1, "softmax", got, 2)
        if got is None:
            return
        if abs(got.sum() - 1.0) > 1e-6:
            return _say(1, False, "", "for the knobs (%g, %g) your two chances add up to %.3f. A row of chances adds up to 1: divide each exponential by the sum of both (line 1b)." % (knobs[0], knobs[1], got.sum()))
        first = 1.0 / (1.0 + np.exp(knobs[1] - knobs[0]))
        want = np.array([first, 1.0 - first])
        if not np.allclose(got, want, atol=1e-6):
            return _say(1, False, "", "for the knobs (%g, %g) the chances should be (%.4f, %.4f) and yours are (%.4f, %.4f). Each chance is e to the power of its knob, divided by the sum of both." % (knobs[0], knobs[1], want[0], want[1], got[0], got[1]))
    lopsided = np.asarray(function(np.array([5.0, -5.0])), dtype=float)
    return _say(1, True, "Knobs (0, 0) give (0.5, 0.5). Knobs (5, -5) give (%.5f, %.5f), a row where one road is almost certain. Every row adds up to 1." % (lopsided[0], lopsided[1]))


def check_play_days(function):
    if not _passed.get(1):
        return _say(2, False, "", "play_days uses your softmax, and Piece 1 has not passed its check yet. Run the check of Piece 1 first.")
    hint = "Each row of dice holds six numbers: the first three choose the roads and the last three are the luck."
    # Part 1: dice written out by hand, six numbers a day. With every knob at 0 each road has chance 0.5.
    # A luck draw of 0.5 gives no luck, and the draws 0.0 and 0.99 give the smallest and the largest luck.
    zeros = np.zeros((7, 2))
    dice = np.array([[0.0, 0.0, 0.0, 0.5, 0.5, 0.5], [0.99, 0.99, 0.99, 0.5, 0.5, 0.5],
                     [0.0, 0.99, 0.0, 0.5, 0.5, 0.5], [0.99, 0.0, 0.99, 0.5, 0.5, 0.5],
                     [0.0, 0.99, 0.0, 0.0, 0.0, 0.0], [0.0, 0.99, 0.0, 0.99, 0.99, 0.99]])
    ok, got = _call(2, "play_days", function, zeros, dice, hint=hint)
    if not ok:
        return
    got = _numbers(2, "play_days", got, len(dice))
    if got is None:
        return
    for row, value in zip(dice, got):
        roads = [0 if d < 0.5 else 1 for d in row[:3]]
        luck = [luck_from(u) for u in row[3:]]
        places = places_of(roads)
        usual = sum(int(COINS[p]) for p in places)
        want = usual + sum(luck)
        if value != want:
            return _say(2, False, "", "with every knob at 0 each road has chance 0.5, so the dice %s mean the roads %s and the luck %s. That day visits %s and pays %d coins: the usual %d plus the luck %d. Your play_days gives %s."
                        % (row.tolist(), roads, luck, _names(places), want, usual, sum(luck), _plain(value)))
    # Part 2: a table with different chances in different places
    tilted = np.zeros((7, 2))
    tilted[0] = [np.log(9.0), 0.0]
    tilted[1] = [0.0, np.log(9.0)]
    dice = np.array([[0.85, 0.0, 0.0, 0.5, 0.5, 0.5], [0.0, 0.3, 0.0, 0.0, 0.0, 0.0]])
    ok, got = _call(2, "play_days", function, tilted, dice, hint=hint)
    if not ok:
        return
    got = _numbers(2, "play_days", got, len(dice))
    if got is None:
        return
    want = _trusted_days(tilted, dice)
    if not np.array_equal(got, want):
        return _say(2, False, "", "with chances 0.9 and 0.1 in Eagle Pass and 0.1 and 0.9 in Mirror Lake, the dice %s and %s should pay %d and %d coins. Yours pay %s and %s. In every step Mara reads the row of the place she stands in, chances[place], takes road 0 when the road dice value is below its first chance, and collects the usual coins of the new place plus the luck."
                    % (dice[0].tolist(), dice[1].tolist(), want[0], want[1], _plain(got[0]), _plain(got[1])))
    # Part 3: many days on random tables, compared with the answer key
    rng = np.random.default_rng(3)
    for trial in range(5):
        table = rng.normal(0.0, 2.0, size=(7, 2))
        dice = rng.random((200, 6))
        ok, got = _call(2, "play_days", function, table, dice, hint=hint)
        if not ok:
            return
        got = _numbers(2, "play_days", got, len(dice))
        if got is None:
            return
        want = _trusted_days(table, dice)
        if not np.array_equal(got, want):
            k = int(np.argmax(got != want))
            return _say(2, False, "", "on a random table the day with the dice %s should pay %d coins and yours pays %s. Check the three lines of the walk." % (np.round(dice[k], 3).tolist(), want[k], _plain(got[k])))
    # Part 4: what the days average, and a table that makes the best path almost certain
    dice = np.random.default_rng(0).random((20000, 6))
    ok, got = _call(2, "play_days", function, zeros, dice, hint=hint)
    if not ok:
        return
    got = _numbers(2, "play_days", got, len(dice))
    if got is None:
        return
    average = float(got.mean())
    if abs(average - exact_J(zeros)) > 0.15:
        return _say(2, False, "", "over 20000 days with every knob at 0 the average should be close to the exact %.2f coins, and yours is %.2f. The luck averages to zero." % (exact_J(zeros), average))
    best = np.zeros((7, 2))
    best[0] = [10.0, -10.0]
    best[1] = [-10.0, 10.0]
    best[4] = [10.0, -10.0]
    ok, got = _call(2, "play_days", function, best, dice[:500], hint=hint)
    if not ok:
        return
    got = _numbers(2, "play_days", got, 500)
    if got is None:
        return
    allowed = 5 * np.sqrt(LUCK * (LUCK + 1) / 500.0) + 0.05
    best_pay = int(path_coins().max())
    if abs(got.mean() - best_pay) > allowed or got.min() < best_pay - 3 * LUCK or got.max() > best_pay + 3 * LUCK:
        return _say(2, False, "", "a table whose knobs make the best path almost certain should give about %d coins a day, between %d and %d with the luck, and yours gives %.2f on average, between %d and %d." % (best_pay, best_pay - 3 * LUCK, best_pay + 3 * LUCK, got.mean(), got.min(), got.max()))
    return _say(2, True, "With every knob at 0 the dice (0.0, 0.99, 0.0) take road 0, road 1, road 0: Mirror Lake, Vine Terraces and Olive Grove Estate, 21 coins without luck. The luck draws 0.0 and 0.99 give the smallest and the largest luck, -%d and +%d coins. Over 20000 days with every knob at 0 your Mara averages %.2f coins, and the exact average is %.2f. A table whose knobs make the best path almost certain pays %.2f on average." % (LUCK, LUCK, average, exact_J(zeros), got.mean()))


def check_measured_gradient(function):
    if not _passed.get(2):
        return _say(3, False, "", "measured_gradient plays days with your play_days, and Piece 2 has not passed its check yet. Run the check of Piece 2 first.")
    w = np.zeros((7, 2))
    # Part 1: with a tiny eps the two tables play identical days when they share the dice, so g is 0
    ok, g = _call(3, "measured_gradient", function, w, (0, 0), 200, 1e-12, np.random.default_rng(1))
    if not ok:
        return
    if np.any(w != 0):
        return _say(3, False, "", "measured_gradient has changed the table w itself. The given lines 3a and 3b work on a copy, w_plus, so that w stays as it is. Did one of them change?")
    g = _one_number(3, "measured_gradient", g)
    if g is None:
        return
    if not np.isfinite(g):
        return _say(3, False, "", "g is %s. Check the division by eps in line 3e." % g)
    if abs(g) > 1e-3:
        if g * 1e-12 < -5:
            return _say(3, False, "", "with an eps of 0.000000000001 the two tables play the same days when they share the dice, so g has to be 0, and yours is %.3g, about minus the average coins divided by eps. Check the parentheses in line 3e: the whole difference J_plus - J is divided by eps." % g)
        return _say(3, False, "", "with an eps of 0.000000000001 the two tables play the same days when they share the dice, luck included, so g has to be 0, and yours is %.3g. The two tables seem to be played with different dice. The first line of the function makes the dice once: give those same dice, dice, to both play_days calls." % g)
    # Part 2: the mean of many measurements is the exact slope
    rng = np.random.default_rng(2026)
    eps = 0.25
    roads = ["road 0", "road 1"]
    for cell in [(0, 0), (0, 1), (4, 0)]:
        bumped = w.copy()
        bumped[cell] = bumped[cell] + eps
        slope = (exact_J(bumped) - exact_J(w)) / eps
        where = "%s of %s" % (roads[cell[1]], PLACES[cell[0]])
        try:
            measured = [function(w, cell, 2000, eps, rng) for _ in range(10)]
        except Exception as error:
            return _say(3, False, "", "measured_gradient stopped with %s: %s, for %s. Look at the lines marked %s in the cell above." % (type(error).__name__, error, where, BULLSEYE))
        mean = float(np.mean(measured))
        if abs(mean - slope) > 0.25 * abs(slope):
            ratio = mean / slope
            if mean == 0.0:
                hint = "Every measurement is exactly 0. Line 3d has to play the copy w_plus; played with w it gives the same average as line 3c."
            elif abs(ratio - eps) < 0.4 * eps:
                hint = "That looks like a missing division by eps."
            elif ratio < -20:
                hint = "Check the parentheses: the whole difference J_plus - J is divided by eps."
            elif ratio < -0.5:
                hint = "The sign is reversed: g is (J_plus - J) / eps."
            elif ratio > 100:
                hint = "That looks like the coins of the days added up. Each J is an average over the days: use .mean()."
            else:
                hint = "Check lines 3d and 3e: the average coins of the copy w_plus with the same dice, then the change J_plus - J divided by eps."
            return _say(3, False, "", "for %s the exact slope is %.3f coins per unit of knob, and the mean of 10 measurements of yours is %.3f. %s" % (where, slope, mean, hint))
        if cell == (0, 0):
            low, high = min(measured), max(measured)
            first_line = "For %s the exact slope is %.2f coins per unit of knob. Your 10 measurements range from %.2f to %.2f and average %.2f." % (where, slope, low, high, mean)
    _say(3, True, first_line + " A single measurement is noisy, and the average of many lands on the exact slope. Road 1 of Eagle Pass and road 0 of Vine Terraces passed the same test.")


def check_learn(function):
    if not _passed.get(3):
        return _say(4, False, "", "learn calls your measured_gradient, and Piece 3 has not passed its check yet. Run the check of Piece 3 first.")
    # Part 1: put a stand-in that always returns g = 2 in place of measured_gradient,
    # so that the choice of the cell and the move of the knob show up without any noise
    calls = []
    real = globals()["measured_gradient"]

    def stand_in(w, cell, n_days, eps, rng):
        calls.append((cell, n_days, eps))
        return 2.0

    globals()["measured_gradient"] = stand_in
    try:
        ok, result = _call(4, "learn", function, 600, 10, 0.1, 0.5, 1,
                           hint="A cell is a tuple (row, column), with a row from 0 to 6 and a column of 0 or 1.")
    finally:
        globals()["measured_gradient"] = real
    if not ok:
        return
    try:
        w, curve = result
        w = np.asarray(w, dtype=float)
    except (TypeError, ValueError):
        return _say(4, False, "", "learn has to return the pair w, curve. Keep the last line of the given code.")
    if len(calls) == 0:
        return _say(4, False, "", "learn never calls measured_gradient. Line 4b has to call it with w, the cell, n_days, eps and rng.")
    if len(calls) != 600:
        return _say(4, False, "", "learn called measured_gradient %d times in 600 iterations. It is called once per iteration." % len(calls))
    for cell, n_days, eps in calls:
        valid = isinstance(cell, tuple) and len(cell) == 2 and 0 <= cell[0] <= 6 and 0 <= cell[1] <= 1
        if not valid:
            return _say(4, False, "", "a cell is a tuple (row, column), with a row from 0 to 6 and a column of 0 or 1. Yours was %s." % (_plain(cell),))
        if n_days != 10 or eps != 0.1:
            return _say(4, False, "", "measured_gradient takes its arguments in this order: w, cell, n_days, eps, rng.")
    total = float(w.sum())
    if abs(total - 600.0) > 1e-6:
        if abs(total - 1200.0) < 1e-6:
            hint = "alpha is missing in the update: the cell moves by alpha * g."
        elif abs(total - 300.0) < 1e-6:
            hint = "g is missing in the update: the cell moves by alpha * g."
        elif total < 0:
            hint = "the cell moves the wrong way: a positive g moves it up."
        elif total == 0:
            hint = "no knob changed: line 4c has to move the chosen cell."
        else:
            hint = "line 4c has to add alpha * g to the chosen cell and to nothing else."
        return _say(4, False, "", "with g = 2 and alpha = 0.5 every iteration should add 1 to one knob, so 600 iterations add 600 in total. Your table adds up to %.1f. %s" % (total, hint))
    counts = w / 1.0
    if counts.min() < 10:
        return _say(4, False, "", "some cells almost never get picked. The table has 7 rows and 2 columns, and the given line 4a picks both the row and the column at random. Has it been changed?")
    if len(curve) != 601:
        return _say(4, False, "", "curve should hold the exact average coins before the first iteration and after each of the 600 iterations, 601 numbers. Yours has %d." % len(curve))
    if abs(curve[0] - exact_J(np.zeros((7, 2)))) > 1e-9 or abs(curve[-1] - exact_J(w)) > 1e-9:
        return _say(4, False, "", "curve has to start with the exact average coins of the starting table and end with that of the final table. Keep the given lines that record it.")
    # Part 2: a real run
    ok, result = _call(4, "learn", function, 200, 100, 0.1, 0.5, 7)
    if not ok:
        return
    w, curve = result
    if curve[-1] < curve[0] + 1.5:
        return _say(4, False, "", "after 200 iterations the exact average coins went from %.2f to %.2f. Every piece passes its check, so look at the order of the arguments in line 4b and at the sign in line 4c." % (curve[0], curve[-1]))
    return _say(4, True, "With a made-up gradient of 2 every iteration moved exactly one cell by alpha * g, and every cell got picked. In a real run of 200 iterations the exact average coins rose from %.2f to %.2f." % (curve[0], curve[-1]))


def pieces_ready():
    missing = [str(n) for n in (1, 2, 3, 4) if not _passed.get(n)]
    if missing:
        print("Not yet. %s %s %s not passed %s check. Finish %s, run %s check cell, and then run this cell again."
              % ("Piece" if len(missing) == 1 else "Pieces", ", ".join(missing), "has" if len(missing) == 1 else "have",
                 "its" if len(missing) == 1 else "their", "it" if len(missing) == 1 else "them", "its" if len(missing) == 1 else "their"))
        return False
    return True


# ======================================================================
# The picture of a run
# ======================================================================

def show_result(w, curve):
    INK, SECONDARY, MUTED, GRID, AXIS = "#0b0b0b", "#52514e", "#898781", "#e1e0d9", "#c3c2b7"
    SURFACE, BLUE = "#fcfcfb", "#2a78d6"
    start, best = curve[0], int(path_coins().max())
    first = first_road_chances(w)
    chances = np.stack([first, 1.0 - first], axis=1)
    best_roads = ALL_PATHS[int(path_coins().argmax())]
    fig, (left, right) = plt.subplots(1, 2, figsize=(11.5, 4.4), gridspec_kw={"width_ratios": [1.5, 1]}, facecolor=SURFACE)

    # the learning curve: the exact average coins after every iteration
    left.set_facecolor(SURFACE)
    left.plot(range(len(curve)), curve, color=BLUE, linewidth=2)
    left.axhline(start, color=MUTED, linewidth=1)
    left.axhline(best, color=MUTED, linewidth=1)
    right_edge = len(curve) * 0.98
    left.text(right_edge, start - 0.12, "start, every knob at 0: %.2f" % start, ha="right", va="top", color=SECONDARY, fontsize=10)
    left.text(right_edge, best + 0.12, "the best path pays %d on average" % best, ha="right", va="bottom", color=SECONDARY, fontsize=10)
    left.plot([len(curve) - 1], [curve[-1]], "o", color=BLUE, markersize=8, markeredgecolor=SURFACE, markeredgewidth=2)
    left.text(right_edge, curve[-1] - 0.45, "now: %.2f" % curve[-1], ha="right", va="top", color=INK, fontsize=10)
    left.set_ylim(start - 1.0, best + 0.8)
    left.set_xlabel("iteration (one hop of the grasshopper)", color=SECONDARY)
    left.set_ylabel("average coins of a day, exact", color=SECONDARY)
    left.set_title("Mara's average coins as the table is learned", color=INK, fontsize=12, loc="left")
    left.grid(True, color=GRID, linewidth=1)
    left.set_axisbelow(True)
    left.tick_params(colors=MUTED)
    for side in ("top", "right"):
        left.spines[side].set_visible(False)
    for side in ("left", "bottom"):
        left.spines[side].set_color(AXIS)

    # the table of chances at the end, one row per place with a choice
    right.set_facecolor(SURFACE)
    blues = LinearSegmentedColormap.from_list("blues", ["#cde2fb", "#256abf", "#0d366b"])
    right.imshow(chances, cmap=blues, vmin=0.0, vmax=1.0, aspect="auto")
    for row in range(7):
        for col in range(2):
            right.text(col, row, "%.2f" % chances[row, col], ha="center", va="center", fontsize=10,
                       color="white" if chances[row, col] > 0.5 else INK)
    place = 0
    for road in best_roads:
        right.add_patch(plt.Rectangle((road - 0.5, place - 0.5), 1, 1, fill=False, edgecolor=INK, linewidth=2))
        place = 2 * place + 1 + road
    right.set_yticks(range(7))
    right.set_yticklabels(PLACES[:7], color=SECONDARY)
    right.set_xticks([0, 1])
    right.set_xticklabels(["road 0\nsouthwest", "road 1\nsoutheast"], color=SECONDARY)
    right.set_xlabel("outlined: the three roads of the best path", color=SECONDARY)
    right.set_title("The table of chances at the end", color=INK, fontsize=12, loc="left")
    right.tick_params(colors=MUTED, length=0)
    for side in ("top", "right", "left", "bottom"):
        right.spines[side].set_visible(False)

    fig.tight_layout()
    plt.show()

    path, probability = most_likely_path(w)
    print("Mara's most likely path after learning:")
    show_path(places_of(path))
    print("She takes it on %.0f%% of her days. The best of the eight paths pays %d coins on average." % (100 * probability, best))
    print("Average coins of a day, exact: %.2f at the start, %.2f now, and at most %d." % (start, curve[-1], best))
    batch = np.asarray(play_days(w, np.random.default_rng(0).random((100, 6))), dtype=float)
    print("A batch of 100 days played with the final table averages %.2f coins. The luck and the dice make a batch differ from the exact average, and every measurement of the method is a batch like this one." % batch.mean())


def compare_dice(learn_function, iterations, n_days, eps, alpha, runs):
    # Run the method with the same dice for both tables and with fresh dice for the copy.
    # measured_gradient plays the table first and its copy second, so every second call of
    # play_days gets a new set of dice in the fresh dice version. Your own code does the rest.
    real = globals()["play_days"]
    best = ALL_PATHS[int(path_coins().argmax())]
    print("%d runs of %d iterations each, seeds 0 to %d" % (runs, iterations, runs - 1))
    for label, fresh in [("the same dice for both tables", False), ("fresh dice for the copy", True)]:
        private = np.random.default_rng(12345)
        calls = [0]

        def play(w, dice):
            calls[0] += 1
            if fresh and calls[0] % 2 == 0:
                dice = private.random(dice.shape)
            return real(w, dice)

        globals()["play_days"] = play
        try:
            found = 0
            finals = []
            for seed in range(runs):
                w, curve = learn_function(iterations, n_days, eps, alpha, seed)
                found += int(most_likely_path(w)[0] == best)
                finals.append(curve[-1])
        finally:
            globals()["play_days"] = real
        print("%-30s the best path is the most likely path in %2d of %d runs. Average coins at the end: median %.2f, lowest %.2f."
              % (label, found, runs, np.median(finals), min(finals)))


print("Ready. Nothing was installed and nothing was downloaded.")

## The road

Nothing to fill in here. The fifteen places are numbered row by row from Eagle Pass, which is place 0. The two roads out of place `i` lead to place `2i + 1` (the southwest road, road 0) and place `2i + 2` (the southeast road, road 1). Places 0 to 6 have a choice. Places 7 to 14 are where a day ends. A path is the way of three roads from Eagle Pass to one of those eight places.

`COINS[i]` is the usual pay of place `i`, what it pays Mara on average. Eagle Pass pays nothing, since nobody arrives there. Every other arrival pays the usual coins plus a luck of `-LUCK` to `+LUCK` coins, all equally likely and drawn anew at every arrival: with `LUCK = 1` the luck is -1, 0 or +1. `luck_from` turns a random number between 0 and 1 into one luck. The simulator reads `COINS` and `luck_from` to pay her. The method works from the total of each day alone, as Mara does in the game.

`exact_J(w)` is given. It adds up the eight paths, each weighted by its chance under the table `w`, and returns the exact average coins of a day. The luck averages to zero, so the usual coins are all it needs. It draws the plot and runs the checks, and the method never calls it.

In [ ]:
# The fifteen places in the order of their numbers, and what each one pays on average when Mara arrives.
PLACES = ["Eagle Pass", "Mirror Lake", "Windmill Market", "Millbrook", "Vine Terraces",
          "Red Canyon", "Dune Inn", "Fishing Harbour", "Reef Lighthouse",
          "Olive Grove Estate", "Citrus Market", "Cliff Monastery", "Hot Springs",
          "Salt Oasis", "Smugglers' Cove"]
COINS = np.array([0, 7, 9, 8, 7, 3, 4, 4, 3, 7, 4, 2, 7, 2, 1])

# The luck of an arrival goes from -LUCK to +LUCK coins. The smallest usual pay is 1,
# so LUCK = 1 is the largest size that never pays a negative amount.
LUCK = 1


def luck_from(u):
    # u is a random number between 0 and 1. Its place among 2 * LUCK + 1 equal parts of that range is the luck.
    return int(u * (2 * LUCK + 1)) - LUCK


show_road()
print()
print("The eight possible paths pay", path_coins().tolist(), "coins on average.")
print("With every knob at 0, Mara tosses a fair coin at every fork. The exact average of her day is", exact_J(np.zeros((7, 2))), "coins.")
print()
show_path([1, 4, 9])    # one path: the three places Mara visits after Eagle Pass

## Piece 1: from knobs to chances

A row of the table has two knobs, `w_row = (w1, w2)`. The softmax turns them into two chances:

$$\text{chance}_i = \frac{e^{w_i}}{e^{w_1} + e^{w_2}}$$

Knobs (0, 0) give the chances (0.5, 0.5). Extreme knobs such as (5, -5) give a lopsided row, about (0.99995, 0.00005), and Mara almost always takes the first road.

🎯 **Piece 1.** Complete `softmax(w_row)`. **1a)** the exponential of each knob (`np.exp` works on both knobs at once). **1b)** divide each exponential by the sum of both, so that the two chances add up to 1.

<details><summary>💡 Hint</summary>

`np.exp(w_row)` is an array of two numbers, and `.sum()` adds them up.
</details>

In [ ]:
def softmax(w_row):
    exps = ...                     # 🎯 1a) e to the power of each knob
    return ...                     # 🎯 1b) divide by the sum of both, so the chances add up to 1

In [ ]:
check_softmax(softmax)

## Piece 2: play days with a table

A day starts at Eagle Pass, place 0, and has three roads. At every road Mara looks at the row of the table that belongs to the place she stands in. Her chance of taking road 0 is the first number of the softmax of that row.

The dice decide. `dice` has one row per day, and each row holds six random numbers between 0 and 1. The first three choose the roads, one for each road of that day: Mara takes road 0 when the dice value is **below** her chance of taking it, and road 1 otherwise. The last three are the luck, one for each arrival: the given line `luck = luck_from(day[3 + step])` turns the luck draw of that arrival into a luck of `-LUCK` to `+LUCK` coins. The dice come in from outside because Piece 3 plays two different tables with the same six numbers a day.

Every arrival pays the usual coins of its place plus the luck.

In every step of the walk, **2a)** is given: the road, 0 or 1, from the dice value and the chance of road 0 (`chances[place][0]` is that chance where Mara stands).

🎯 **Piece 2.** Complete the other two lines of the step. **2b)** the next place: the roads out of place `i` lead to places `2i + 1` and `2i + 2`. **2c)** the coins in the hat, which now include the usual coins of the place Mara has just reached and her luck there.

<details><summary>💡 Hint</summary>

Road 0 leads to place `2i + 1` and road 1 to place `2i + 2`, so the next place is `2 * place + 1 + road`. Add `COINS[place] + luck` after the place has changed.
</details>

In [ ]:
def play_days(w, dice):
    chances = [softmax(row) for row in w]       # chances[i] holds the chances of road 0 and road 1 in place i
    totals = []
    for day in dice:                            # one day is six dice values: three for the roads, three for the luck
        place = 0                               # every day starts in Eagle Pass
        total = 0                               # the coins in the hat so far
        for step in range(3):                   # a day is three roads long
            road = 0 if day[step] < chances[place][0] else 1   # 2a) road 0 when the dice value is below its chance, else road 1
            place = ...                                        # 🎯 2b) the next place
            luck = luck_from(day[3 + step])                    # the luck of this arrival comes from the dice of this step
            total = ...                                        # 🎯 2c) add the usual coins of the place just reached and the luck
        totals.append(total)
    return np.array(totals)                     # the coins of every day

In [ ]:
check_play_days(play_days)

## Piece 3: the measured gradient of one cell

The recap's `grad(f, x)` computed `(f(x + eps) - f(x)) / eps`. Here `f` is the average coins of a table and `x` is one cell, given as a pair `cell = (row, column)`. While we play, `f` cannot be computed exactly, so we estimate it by playing `n_days` days:

- `J` is the average coins of `n_days` days played with the table `w`;
- `J_plus` is the average coins of the same `n_days` days played with `w_plus`, a copy of `w` in which `eps` is added to the chosen cell;
- the measured gradient is `g = (J_plus - J) / eps`.

Both tables are played with the same dice, which the function makes once at its top. Both tables also get the same luck on every day, so the luck cancels in `J_plus - J`, and any difference comes from the change in the cell alone. The dice still choose the roads, so one measured gradient is noisy: it is right on average and off by some amount each time. The loop of Piece 4 repeats it many times.

The first three lines of the function are given. **3a)** makes a copy of the table with `w.copy()`. Without the copy, `w_plus = w` would give the same array a second name, and adding `eps` to one would change the other. **3b)** adds `eps` to the chosen cell of the copy. **3c)** is the average coins of the table: `play_days(w, dice)` returns one number per day, and `.mean()` averages them.

🎯 **Piece 3.** Complete the last two lines. **3d)** the average coins of the copy, with the same dice. **3e)** the measured gradient.

<details><summary>💡 Hint</summary>

Line 3c plays the table `w` with `dice` and averages the days. Play the copy `w_plus` the same way, with the same `dice`. The measured gradient is the change `J_plus - J`, divided by `eps`.
</details>

In [ ]:
def measured_gradient(w, cell, n_days, eps, rng):
    dice = rng.random((n_days, 6))                  # one set of dice for both tables: three for the roads, three for the luck
    w_plus = w.copy()                               # 3a) a copy of the table, so that w itself stays as it is
    w_plus[cell] = w_plus[cell] + eps               # 3b) add eps to the chosen cell of the copy
    J = play_days(w, dice).mean()                   # 3c) average coins of the table
    J_plus = ...                                    # 🎯 3d) average coins of the copy, with the same dice
    g = ...                                         # 🎯 3e) change in altitude over step size
    return g

In [ ]:
check_measured_gradient(measured_gradient)

## Piece 4: the method

Now the loop. Every iteration is one hop of the grasshopper: pick a cell, measure its gradient by playing, move the cell. The frame of the loop, the choice of the cell and the recording of the exact average coins (for the plot) are given.

**4a)** is given: a cell is a pair `(row, column)` with a row from 0 to 6 and a column of 0 or 1, and both are picked at random with `rng.integers(...)`.

🎯 **Piece 4.** Complete the other two steps. **4b)** the measured gradient of that cell, from Piece 3. **4c)** move the cell: `cell = cell + alpha * g`, which moves it up when `g` is positive and down when `g` is negative.

<details><summary>💡 Hint</summary>

`measured_gradient` takes `w`, the `cell`, `n_days`, `eps` and `rng`, in this order. The table `w` is a numpy array, and `w[cell]` reads and writes the knob at `cell`.
</details>

In [ ]:
def learn(iterations, n_days, eps, alpha, seed):
    rng = np.random.default_rng(seed)       # the random cells and the dice come from here
    w = np.zeros((7, 2))                    # seven rows (the places with a choice), two columns (the roads), every knob at 0
    curve = [exact_J(w)]                    # the exact average coins before the first iteration
    for iteration in range(iterations):
        cell = (rng.integers(7), rng.integers(2))               # 4a) pick one cell at random: (row, column)
        g = ...                                                 # 🎯 4b) measure its gradient by playing
        w[cell] = ...                                           # 🎯 4c) move the cell
        curve.append(exact_J(w))            # the exact average coins after this iteration, for the plot
    return w, curve

In [ ]:
check_learn(learn)

## Run it and look

Nothing to fill in. Mara starts with every knob at 0 and learns by playing. In every iteration she plays `N_DAYS` days with the table and `N_DAYS` days with the copy. The plot uses `exact_J` after every iteration to show the true average coins of her table. The method itself never sees it.

The numbers below are yours to change. Run the cell again after a change.

In [ ]:
N_DAYS = 100         # days played with each table, in every iteration
EPS = 0.1            # how far the chosen cell is nudged to measure its gradient
ALPHA = 0.5          # how far a cell jumps per unit of gradient
ITERATIONS = 300     # how many hops
SEED = 0             # seeds the random cells and the dice, so that a run can be repeated

if pieces_ready():
    print("This run plays %d days." % (2 * N_DAYS * ITERATIONS))
    w, curve = learn(ITERATIONS, N_DAYS, EPS, ALPHA, SEED)
    show_result(w, curve)

## What you just saw

The grasshopper climbed. Its altitude is the average coins of a day, and it rose from 17.25 to nearly 21, the average pay of the best path. The table of knobs is the place where the grasshopper stands: fourteen numbers, so this hill has fourteen directions. In every iteration the grasshopper looked along one direction (the random cell), measured the slope there by playing days (`eps` added to the cell, the same dice for both tables), and jumped `alpha * g` along it.

Nobody told Mara what a place pays, and every day paid a little differently. She saw the coins of the days she played, and that was enough.

**Try it.** Change a number in the cell of "Run it and look" and run it again.

- `ALPHA = 0.01`: the jumps are tiny, and after 300 iterations the average coins have hardly moved.
- `EPS = 0.001`: with the same dice the two tables play almost the same days, so about 99 measurements in 100 give g = 0 and the others give a huge g.
- `N_DAYS = 5`: every measurement is rough, and some seeds end on a worse path. Try several values of `SEED`.

The next section is optional. It compares the same dice with fresh dice.

## ⭐ Optional: why the same dice

In `measured_gradient` both tables are played with the same dice. The cell below runs the method 20 times that way, and 20 times with fresh dice for the copy (a second `rng.random((n_days, 6))`), and counts the runs that end with the best path as the most likely path. It takes a few seconds.

With the same dice both tables also get the same luck on every day, so the luck cancels in `J_plus - J`. With fresh dice the luck of two different sets of days stays in the difference, on top of the change in the cell. The measured gradient is still right on average, and a single measurement is several times as spread out.

In [ ]:
if pieces_ready():
    compare_dice(learn, ITERATIONS, N_DAYS, EPS, ALPHA, 20)